[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/travisormsby/python-data/blob/main/notebook/data_therapy.ipynb)

# Python data therapy

Sometimes, when your code doesn't work the way you think it should, it's because of the way you're representing data. This workshop covers three modern Python libraries for representing data:

1. Use `Pydantic` to enforce a data contract
1. Use `Polars` to analyze a data set that fits in memory
1. Use `DuckDB` to analyze a data set too large for memory

## Google Colab setup

If you are running this notebook in Colab, run the cell below to set it up to run correctly. Skip this step if you followed the instructions in the README to run the notebook locally.

In [ ]:
%pip install geojson_pydantic
!git clone https://github.com/travisormsby/python-data

import os

os.chdir("python-data/notebook")

## Pydantic

One way that your data representation can cause problems is if you aren't able to guarantee that it has the correct information. 

For example, let's say you have a tuple of values that represent the population, total area, and water area for a census tract.

In [ ]:
tract = (3000, 4.0, 1.0)

And a function that computes the population density of a tract per unit land area.

In [ ]:
def pop_density(tract):
    land_area = tract[1] - tract[2]
    return tract[0]  / land_area


pop_density(tract)

There are lots of ways for this to go wrong. For example, what if an attribute has an unexpected type?

In [ ]:
tract = (3000, 4.0, "1.0")
pop_density(tract)

### Parsing data

These types of problems are especially common at the boundaries of your code, when you are interacting with data that came from somewhere else. What you want is a way to be sure up front that any data you're working with has the information you expect it to have.

Pydantic is a library for parsing data. Parsing means getting data into a form that you know is correct.

This requires you to know the correct form, which is not trivial. It may not always be possible. Or maybe not possible with a reasonable amount of effort. It's a constraint that you balance against other constraints. But getting your data into a form you know is correct should have a relatively high priority. 

Pydantic can create what's called a model with all the fields the data should have, and the acceptable data type for each field. Most models work by inheriting from the `BaseModel` class.

In [ ]:
from pydantic import BaseModel


class Tract(BaseModel):
    pop: int
    area: float
    water: float

Instances of the `Tract` model are constructed differently from raw tuples in that you must specify the name of the field for each value. 

Pydantic models will also coerce values to the correct type if possible.

In [ ]:
tract = Tract(pop=3000, area=4.0, water="1.0")
print(tract)

If you don't have the right information, Pydantic will fail when creating the instance, throwing a `ValidationError`. Contrast that with the broken tuple that only failed when it was passed to the `pop_density` function. That's good because if something is going to fail, you want it to fail fast.

In [ ]:
from pydantic import ValidationError

try:
    tract = Tract(pop="3000", area="4")
    print(tract)
except ValidationError as e:
    print(e)

The `pop_density` function can be rewritten to take an instance of the `Tract` model instead of a tuple. Because the values are accessed by name, instead of by index number, the function is also much more clear.

In [ ]:
def pop_density(tract):
    land_area = tract.area - tract.water
    return tract.pop / land_area

pop_density(tract)

### Advanced parsing

Pydantic can do more than just ensure the right values with the right data types are present. The `Field` class can make sure the values are within the correct range, and provide default values for a field. or example, to ensure that the `pop` and `water` are not negative, and that the `area` field is greater than zero. If any of the values are wrong, Pydantic will let you know about all of the wrong values, not just fail on the first wrong one.

In [ ]:
from pydantic import Field


class Tract(BaseModel):
    pop: int = Field(ge=0)
    area: float = Field(gt=0)
    water: float = Field(default=0, ge=0)

try:
    tract = Tract(pop=-1, area=0)
    print(tract)
except ValidationError as e:
    print(e)

You can also use the `model_validator` to validate fields against each other. For example, ensuring that `water` is not larger than `area`.

In [ ]:
from pydantic import model_validator


class Tract(BaseModel):
    pop: int = Field(ge=0)
    area: float = Field(gt=0)
    water: float = Field(default=0.0, ge=0)

    @model_validator(mode="after")
    def check_water(self):
        if self.water > self.area:
            raise ValueError("Water area cannot be larger than total area")
        return self

try:
    tract = Tract(pop=3000, area=4.0, water=5.0)
    print(tract)
except ValidationError as e:
    print(e)

### Instantiating models from input, and back

If your data is already in a dictionary whose keys correspond to model fields, you can use the `model_validate` method to create an instance of the model from that dictionary.

In [ ]:
tract_data = {"pop": 3000, "area": 4.0}
tract = Tract.model_validate(tract_data)
print(tract)

The `model_validate_json` method does the same thing for json strings, such as those you might get from a REST endpoint.

In [ ]:
tract_json = '{"pop": 3000, "area": 4.0}'
tract = Tract.model_validate_json(tract_json)
print(tract)

The `model_dump` and `model_dump_json` methods let you go the other direction.

In [ ]:
tract.model_dump()

In [ ]:
tract.model_dump_json()

One powerful use case of being able to create models directly from data is the `geojson_pydantic` library, which allows you to parse a string into a model to be sure it adheres to the GeoJSON standard.

In [ ]:
from geojson_pydantic import Point

geojson = """{
    "type": "Point",
    "coordinates": [-95, 45]
}"""

point = Point.model_validate_json(geojson)
print(point)

### Exercise: Create Pydantic models

- Option 1: On the [Python data problems](https://projects.travisormsby.com/python-data) page, do problems 1 and 2. These problems give you all the lines of code, and you drag and drop them into the right order until the code runs successfully. Then answer these questions:

    1. What kind of values could be coerced to `int` in problem 1? Which could not be?
    1. Problem 1 directly instantiated models, while problem 2 read the data from a file. Which approach is better for schema that are regularly changing?
    1. Problem 2 looped over every row in the CSV. How well would that have worked with 100 million row csv?
- Option 2: Write a script from scratch that creates a Pydantic model that matches the schema in `data/mn_counties.csv` and then creates instances of the model for each row in that file.


## Polars

Another way your data representation can cause problems is that some ways of representing data make your code slow. The two biggest sources of unnecessary slowdown are data representations that require:

- Using Python loops for processing
- Looking at all the data when the question only requires a subset of it

You may already know about a library called Pandas that represents tabular data in a way that can solve the first problem. Polars is similar to Pandas, but better in that it can solve both of these problems.

### Dataframes

You can read a variety of file types into a Polars dataframe. It is especially good for reading Parquet files because Polars represents data in memory using the Apache Arrow specification, which is the same way Parquet files were designed to be read. This means the data usually don't have to be transformed after reading them into memory from the file, making file I/O faster.

In [ ]:
import polars as pl

df = pl.read_parquet("data/mn_counties.parquet")
display(df)

Use the `col` function to specify a column by name. This is the most common way to create what's called an expression, which are the fundamental unit of work in Polars. 

In [ ]:
pop = pl.col("population")
area = pl.col("area_km2")
region = pl.col("region")

Importantly, expressions don't actually do anything by themselves. They are like recipes, not cooking.

In [ ]:
pop

To do any work, expressions need a context. For example, use the `select` context when you want just the results of the expressions you specify.

In [ ]:
df.select(pop)

The same expression produces different results in different contexts. For example, use the `with_columns` context when you want all the columns of the original table and also the results of the expressions you specify. If any of the columns you specify have the same name as a column in the dataframe, the original columns are replaced with the result of the expression.

In the example below, the `population` column already exists, so it is overwritten with the values in the `pop` expression.

In [ ]:
df.with_columns(pop)

### Vectorization

You can use Polars expressions to batch operations over an entire group of values. That's good, because otherwise you would need to write a slow Python loop to process each value one at a time. This batching is called "vectorization", and can make code execution 100x faster (or more).

For example, if you wanted to find the counties with population over 100,000 people, you wouldn't loop over every row in the table and check whether its population met the threshold. You would use a logical operator to compare the entire `population` column to the threshold.

Vectorized logical operations like this are very useful in the `filter` context, which filters the rows based on expressions that return a boolean value.

In [ ]:
large_counties = pop > 100000
df.filter(large_counties)

In addition to logical operators, you can use entire columns in vectorized arithmetic operations. For example, if you want a column of population density, you can divide the `population` column by the `area_km2` column.  

In [ ]:
pop_density = pop / area
df.with_columns(pop_density)

Notice that this expression put the population density values into the `population` column. That's bad both because it overwrote data you might still want and also because the data have been transformed in such a way that `population` is not a good name for those values.

This happened because an expression's default name is based on the "left hand rule" - it keeps the name of the first column in the expression. Because the `pop` expression's name is `population`, it replaced the original `population` column in the `with_columns` context. To prevent problems like this, it is common to change an expressions name using the `alias` method.

In [ ]:
pop_density = (pop / area).alias("pop_density")
df.with_columns(pop_density)

Other vectorized expression methods reduce data, like `sum`, which can be very useful in the last major context: `group_by`. This context always includes two parts:

- Grouping rows based on shared values for one or more expressions. Usually this is just a column name.
- Aggregating the values in each group's rows. Usually this is an expression that summarizes values to a single value.

In the example below, all the rows are grouped by region. Within each region group, the `regional_pop` expression sums the values of the `population` column. The summed value for each group is put in a new column with the same name as the expression. Because the `regional_pop` expression was constructed from the `population` column, it's name is `population`.

In [ ]:
region_pop = pop.sum()
df.group_by(region).agg(region_pop)

### Structural transformations

In addition to contexts and expressions, Polars also has structural transformations. These change the shape or structure of the entire table.

For example, the `sort` structural transformation sorts the rows based one the values in one or more columns.

In [ ]:
df.sort(pop, descending=True)

Besides `sort`, another useful structural transformation is `head`, which returns just the top few rows of the table


In [ ]:
df.sort(pop, descending=True).head(3)

### Method chaining

In Polars, the conventional pattern is to chain expression methods, contexts, and structural transformations together. To prevent these long chains from becoming unreadable, you should:

- Pull all but the most trivial expressions out into separate variables
- Compose complex expressions from simpler expressions
- Format each context and structural transformation method call onto its own line.

The example below shows a well-formatted example combining several elements to return a table with the top 3 regions in Minnesota by population density, excluding the Twin Cities.

In [ ]:
region_pop = pop.sum()
region_area = area.sum()
region_density = (region_pop / region_area).alias("density")

df = (
    pl.read_parquet("data/mn_counties.parquet")
    .group_by(region)
    .agg(region_density)
    .sort(pl.col("density"), descending=True)
    .filter(region != "Twin Cities")
    .head(3)
)

display(df)

### Lazyframes

On its face, using an expression inside a context seems seems needlessly complicated. But the idea that expressions are plans (not work) is vital for using Polars effectively. 

So far, these examples have created a dataframe. In Polars, dataframes are eager. Data are immediately read into memory when you create a dataframe. The methods you call on a dataframe are executed immediately. Polars dataframes broadly correspond to how Pandas dataframes work.

But one of the big advantages of Polars is the lazyframe. Lazyframes, like expressions, aren't work. They are a plan for doing work. Data aren't immediately read into memory when you create a lazyframe. The methods you call on them aren't immediately executed. Instead, the work is done only when you materialize the lazyframe (called collection in Polars). That's usually better because Polars can optimize all those operations.

As an analogy, imagine you're grocery shopping. A dataframe is like thinking about what you need one item at a time, and grabbing each item as you think about it, no matter where it is in the store. A lazyframe is like making a grocery list. Collecting the lazyframe is like organizing your list by item location, then doing the shopping based on that organized list.

You can convert a dataframe to a lazyframe with the `lazy` method. But if you don't collect it, you just get a query plan (the grocery list), not the data (the groceries).

In [ ]:
df.lazy()

You can also create a lazyframe when you first read data by using one of the `scan_*` functions instead of the `read_*` functions.

In [ ]:
lf = (
    pl.scan_parquet("data/mn_counties.parquet")
    .group_by(region)
    .agg(region_density)
    .sort(pl.col("density"), descending=True)
    .filter(region != "Twin Cities")
    .head(3)
)

lf.collect()

The naive query plan just runs everything in order. It represents what would have happened if this was a dataframe.

In [ ]:
display(lf)

But the naive query plan is not what Polars actually does when it collects the lazyframe. To help you understand how Polars processes the query, lazyframes have `explain` and `show_graph` methods that illustrate the optimized query plan. 

In [ ]:
lf.show_graph()

In this case, it made three optimizations relative to what would have happened in a dataframe:

1. It pushed the filter to the very beginning, to avoid wasting computing grouping and aggregating Twin Cities rows that were just going to be thrown out.
2. It only read 3 of the 5 columns from the file, to avoid wasting compute processing the unused `name` and `fips` columns that were just going to be thrown out.
3. It used a more efficient sorting algorithm that doesn't track anything other than the top 3 results, to avoid wasting compute sorting rows that would just be thrown out when returning `head(3)`.

### Exercise: Analyze data with Polars

- Option 1: On the [Python data problems](https://projects.travisormsby.com/python-data) page, do problems 3, 4, and 5. Then answer these questions:
    - Which problems used dataframes? Which used lazyframes? How do you know?
    - In problem 3, why is it smarter to use the `filter` context than the `group_by` context to get the Twin Cities' population?
    - In problem 5, what happens if you move the call to the `tail` method earlier? Why?
    
- Option 2: Use Polars and the data in the `data` directory to answer any (or all) of these questions:
    - What is the total area of Minnesota's Central region in square kilometers?
    - What is the smallest Minnesota region by area?
    - What is the population density of Minnesota?



## DuckDB

A third way data representations may cause problems is by failing to interface well with the source data on disk. That's especially true if either: 
- The data is not on your local machine, but is accessed through a network share or a cloud service
- The data is larger than memory, requiring out-of-core processing

DuckDB is an analytical database engine designed to turn nearly any kind of tabular data into a queryable format, without having to load it into an RDBMS like PostgreSQL or SQL Server.

### The Relational API

DuckDB's Relational API relies on two objects:

- `DuckDBPyConnection` (connection): The connection to the database created by `duckdb.connect`. When called without any arguments, the connection is to the default in-memory database that DuckDB creates. It's also possible to specify a DuckDB Database file on disk to connect to.

- `DuckDBPyRelation` (relation): Conceptually identical to a Polars lazyframe. It is a query plan that returns results only when it is materialized. You can materialize a relation in a variety of ways, such as passing it to the `print` function, calling the `show` method, or exporting it to some other format.

In [ ]:
import duckdb

with duckdb.connect() as con:
    rel = (
        con.read_parquet("data/mn_counties.parquet")
        .filter("region = 'Central'")
        .select("name", "area_km2", "population")
    )
    
    rel.show()

DuckDB alternatively supports writing SQL directly for people who are more comfortable with SQL than with Python.

In [ ]:
import duckdb

with duckdb.connect() as con:
    rel = con.sql("""
        SELECT name, area_km2, population
        FROM 'data/mn_counties.parquet'
        WHERE region = 'Central'
    """)

    rel.show()

DuckDB is really good at reading hive partitioned Parquet, because it can skip reading entire files for queries that take advantage of the partitioning. Polars and Pandas can do this too, but DuckDB's query explainer makes it easier to see.

In [ ]:
import duckdb

with duckdb.connect() as con:
    rel = con.sql("""
        SELECT name, area_km2, population
        FROM 'data/mn_counties/**/*.parquet'
        WHERE region = 'Central'
    """)

    rel.explain()

### Preventing SQL injection

SQL injection is a type of security exploit that involves hijacking dynamically constructed SQL statement. To avoid SQL attacks, never use f-strings or string formatting to dynamically build SQL expressions.

For example, if `region` is a user-supplied value that is supposed to limit the query to a single region, a malicious attacker can use SQL injection to get all the values.

In [ ]:
region = "Central' OR 1=1;--"

with duckdb.connect() as con:
    rel = con.sql(f"""
        SELECT name, area_km2, population
        FROM 'data/mn_counties/**/*.parquet'
        WHERE region = '{region}'
    """)
    
    rel.show()

You can get avoid the risks of SQL injection by interacting directly with the `DuckDBPyConnection` using the DB API to call the `execute` function. This lets you efficiently use  [prepared statements](https://duckdb.org/docs/lts/clients/python/dbapi#prepared-statements) to parameterize queries. 

One major disadvantage of the DB API is that it is not lazy the way the Relational API is. But it is the easiest way to avoid SQL injection attacks.

In [ ]:
region = "Central' OR 1=1;--"

with duckdb.connect() as con:
    query = """
        SELECT name, area_km2, population
        FROM 'data/mn_counties/**/*.parquet'
        WHERE region = $region
    """

    params = {"region": region}

    con.execute(query, parameters=params)
    
    df = con.pl()

display(df)

### DuckDB vs. Polars

DuckDB has two major advantages over Polars. The first is that DuckDB is better at handling data that's bigger than memory. DuckDB always spills this kind of data to disk, so you don't get an error saying you ran out of memory. Polars is getting better at this kind of out-of-core processing, but large datasets can cause Polars to crash if you're not careful.

The second advantage is that DuckDB can handle spatial data better than Polars. DuckDB has a built-in geometry datatype that lets it read spatial data from formats like GeoParquet. And it has a spatial extension that provides functions for manipulating geometry. This lets you do things like select based on a spatial predicate instead of only attributes.

In [ ]:
distance = 1000
location = "POINT(570000 5185000)"

with duckdb.connect() as con:
    con.execute("INSTALL spatial; LOAD spatial;")

    query = ("""
        SELECT name_of_program, city
        FROM 'data/child_care.gpkg'
        WHERE ST_Distance(Shape, ST_GeomFromText($location)) < $distance
    """)

    params = {"distance": distance, "location": location}

    con.execute(query, parameters=params)
    
    df = con.pl()

display(df)

The advantage of Polars over DuckDB is that Polars expressions can be built up dynamically and composed together more easily and safely than SQL statements.

Fortunately, these advantages don't end up trading off, because DuckDB and Polars work well together. DuckDB makes use of the same Apache Arrow memory model that Polars does, which makes passing data between the two very efficient. A good general pattern is to use DuckDB to pull in large datasets (using its superior out-of-core data processing), then use Polars for more sophisticated manipulation (using its superior dynamic expressions).

### Exercise: Query data with DuckDB

- Option 1: On the [Python data problems](https://projects.travisormsby.com/python-data) page, do problems 6, 7, and 8. Then answer these questions:
    - In problem 6, why does the order of `filter` and `select` matter?
    - In which problems, if any, did hive partitioning by region make the query more efficient?
    - In problem 8, was the query evaluated lazily or eagerly? How do you know?

- Option 2: Use DuckDB and the data in the `data` directory to answer any (or all) of these questions:
    - How many counties are in the Northland region?
    - How many counties with under 50,000 people are in each region?
    - How many child care centers are inside the polygon with this WKT representation: 
    
        `'POLYGON((500000 5000000, 501000 5000000, 501000 5001000, 500000 5000000, 500000 5000000))'`
        
        Hint: You will want to look at the the [`ST_Within`](https://duckdb.org/docs/lts/core_extensions/spatial/functions#st_within) and [`ST_GeomFromText`](https://duckdb.org/docs/lts/core_extensions/spatial/functions#st_geomfromtext) functions in the DuckDB spatial extension.

## Putting it all together

The Overture Maps foundation has large datasets of geographic features that it stores in the cloud as hive-partitioned GeoParquet. To let a user know the top 5 most common place categories within a polygon they define, you can:

1. Use Pydantic to parse the user-defined GeoJSON polygon to make sure it's valid.

In [ ]:
from geojson_pydantic import Polygon

geojson = """{
    "type": "Polygon",
    "coordinates": [[[-95, 45], [-94, 45], [-94, 46], [-95, 46], [-95, 45]]]
}"""

user_polygon = Polygon.model_validate_json(geojson)

2. Use Shapely to construct a geometry from the Polygon model. Extract the WKB and bounding box coordinates from that geometry and use them for the query parameters. This is guaranteed to work because the GeoJSON dumped by the model has already been parsed and determined to be correct.

In [ ]:
from shapely.geometry import shape

geom = shape(user_polygon.model_dump())
xmin, ymin, xmax, ymax = geom.bounds

params = {
    "user_polygon_wkb": geom.wkb,
    "xmin": xmin,
    "ymin": ymin,
    "xmax": xmax,
    "ymax": ymax
}

3. Construct a SQL query to get the category of every feature. The query uses bboxes to skip most of the Parquet files and parameterized queries to avoid SQL injection.

In [ ]:
query = """
    SELECT basic_category
    FROM 's3://overturemaps-us-west-2/release/2026-08-19.0/theme=places/type=place/*.parquet'
    WHERE bbox.xmin BETWEEN $xmin AND $xmax
      AND bbox.ymin BETWEEN $ymin AND $ymax
      AND ST_Within(geometry, ST_GeomFromWKB($user_polygon_wkb))
"""

4. Use the spatial and httpfs DuckDB extensions to read the remote GeoParquet files using the query and dump them to a Polars dataframe.

In [ ]:
import duckdb

with duckdb.connect() as con:
    con.execute("INSTALL spatial; LOAD spatial;")
    con.execute("INSTALL httpfs; LOAD httpfs;")
    con.execute(query, parameters=params)
    df = con.pl()

display(df)

5. Use the nicely composable Polars expressions to do the final analysis.

In [ ]:
import polars as pl

count_by_category = pl.col("basic_category").len().alias("count")

lf = (
    df.lazy()
    .group_by(pl.col("basic_category"))
    .agg(count_by_category)
    .drop_nulls()
    .sort("count", descending=True)
    .head(5)
)

lf.collect()